In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report
import pickle


In [2]:
import pandas as pd

# Load the dataset
data = pd.read_csv('train.csv')  # Replace 'train.csv' with the actual file name if needed

# Clean column names (strip any unwanted spaces or quotes)
data.columns = data.columns.str.strip().str.replace("'", "")

# Select relevant columns: 'type' for labels and 'text' for message content
data = data[['type', 'text']]  # Select the correct columns

# Rename columns for consistency
data.columns = ['label', 'message']  # Rename 'type' to 'label' and 'text' to 'message'

# Encode the labels (e.g., ham = 0, spam = 1)
data['label'] = data['label'].map({'ham': 0, 'spam': 1})

# Display the first few rows to confirm preprocessing
print(data.head())


   label                                            message
0      0                              Go until jurong point
1      0                     Ok lar... Joking wif u oni...'
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0                   Nah I don't think he goes to usf


In [16]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download necessary NLTK resources (run once)
import nltk
nltk.download('punkt')
nltk.download('stopwords')

# Preprocess the text
def preprocess_text(text):
    # Convert to string, in case there are any non-string values (like NaN, float, etc.)
    text = str(text)
    
    # Convert text to lowercase
    text = text.lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize the text
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    return ' '.join(tokens)

# Apply preprocessing to the 'message' column
data['message'] = data['message'].apply(preprocess_text)
print(data.head())


   label                                            message
0      0                                    go jurong point
1      0                            ok lar joking wif u oni
2      1  free entry wkly comp win fa cup final tkts st ...
3      0                u dun say early hor u c already say
4      0                            nah dont think goes usf


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\spoor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\spoor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\spoor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report
import joblib  # For saving the model
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download necessary NLTK resources (run once)
nltk.download('punkt')
nltk.download('stopwords')

# Load the dataset
data = pd.read_csv('enron_combined_datase.csv')  # Replace 'train.csv' with your actual dataset file name

# Clean column names (strip any unwanted spaces or quotes)
data.columns = data.columns.str.strip().str.replace("'", "")

# Select relevant columns: 'label' for labels and 'message' for message content
data = data[['label', 'text']]  # Select the correct columns

# Rename columns for consistency
data.columns = ['label', 'message']  # Rename 'type' to 'label' and 'text' to 'message'

# Encode the labels (ham = 0, spam = 1)
data['label'] = data['label'].map({'ham': 0, 'spam': 1})

# Handle missing data
data = data.dropna(subset=['message'])  # Remove rows with missing values in 'message'
data['message'] = data['message'].fillna("empty message")  # Replace NaN with a placeholder

# Preprocess the text
def preprocess_text(text):
    # Convert to string, in case there are any non-string values
    text = str(text)
    
    # Convert text to lowercase
    text = text.lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize the text
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    return ' '.join(tokens)

# Apply preprocessing to the 'message' column
data['message'] = data['message'].apply(preprocess_text)

# Vectorize text using TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=3000)  # Use up to 3000 most frequent words
X = tfidf_vectorizer.fit_transform(data['message']).toarray()  # Features
y = data['label']  # Labels

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the SVM model
svm_model = SVC(kernel='linear')  # Linear kernel for SVM
svm_model.fit(X_train, y_train)

# Evaluate the model
y_pred = svm_model.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Save the trained model and TF-IDF vectorizer
joblib.dump(svm_model, 'spam_classifier_model.pkl')  # Save the SVM model
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')  # Save the TF-IDF vectorizer

print("Model and vectorizer saved successfully!")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\spoor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\spoor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3282
           1       0.98      0.99      0.99      3461

    accuracy                           0.99      6743
   macro avg       0.99      0.99      0.99      6743
weighted avg       0.99      0.99      0.99      6743

Model and vectorizer saved successfully!


In [10]:
from sklearn.utils import resample

# Separate the spam and ham emails
spam_df = df[df['type'] == 'spam']
ham_df = df[df['type'] == 'ham']

# Upsample the minority class (spam) to match the number of ham emails
spam_upsampled = resample(spam_df, replace=True, n_samples=len(ham_df), random_state=42)

# Combine the upsampled spam and ham emails
df_balanced = pd.concat([spam_upsampled, ham_df])

# Shuffle the balanced dataset
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Check the balanced dataset
print("Balanced dataset shape:", df_balanced.shape)


Balanced dataset shape: (526, 3)
